# SuperAnimal-Quadruped pose vs snippet (Gradio)

Companion to ``pose_snippet_gradio_viewer.ipynb`` for **DeepLabCut SuperAnimal** outputs:
pose arrays **(35, 39, 3)** with **x, y in snippet pixel space** and likelihood — skeleton indices follow the **SuperAnimal unified quadruped vocabulary** ([modelzoo-figures](https://github.com/AdaptiveMotorControlLab/modelzoo-figures/blob/main/data/superquadruped_dataset.json)). After pulling fixes, **re-run** ``06_pose_extraction_superanimal.py`` so `.npy` keypoint order matches (HDF column order ≠ index order).

Uses ``pose_extraction_superanimal`` from ``dataset_construction/config.yaml`` (output dir + sampling ``target_fps``).

**Requires:** ``pip install gradio`` (and ``opencv-python``, ``numpy``, ``pyyaml``).

Run all cells, then **“Load random snippet + pose”**.

In [1]:
from __future__ import annotations

import json
import os
import random
import tempfile
from pathlib import Path

import cv2
import numpy as np
import yaml


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(14):
        if (p / "dataset_construction" / "config.yaml").is_file():
            return p
        p = p.parent
    raise FileNotFoundError("Run from repo root or a subfolder (need dataset_construction/config.yaml).")


REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "dataset_construction" / "config.yaml"
with open(CONFIG_PATH, encoding="utf-8") as f:
    _cfg = yaml.safe_load(f)
_fd = _cfg.get("final_dataset") or {}
_pe = _cfg.get("pose_extraction_superanimal") or {}
POSE_OUT = REPO_ROOT / str(_pe.get("output_dir", "dataset/embeddings/pose/superanimal_v2"))
POSE_TARGET_FPS = float(_pe.get("target_fps", 5.0))
N_KEYPOINTS = int(_pe.get("n_keypoints", 39))
MANIFEST_PATH = REPO_ROOT / _fd["output_manifest"]

rows: list[dict] = []
with open(MANIFEST_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"Manifest: {MANIFEST_PATH.relative_to(REPO_ROOT)} ({len(rows)} rows)")
print(f"SuperAnimal pose dir: {POSE_OUT.relative_to(REPO_ROOT)}")
print(f"Pose sampling (target_fps): {POSE_TARGET_FPS} Hz")

Manifest: dataset_construction/manifests/final_dataset_v2.jsonl (7318 rows)
SuperAnimal pose dir: dataset/embeddings/pose/superanimal_v2
Pose sampling (target_fps): 5.0 Hz


In [2]:
def _is_superanimal_pose_file(path: Path) -> bool:
    """Shape (T, N_KEYPOINTS, 3). Skips manifest ViTPose paths (17 kp) after a dual extraction."""
    try:
        a = np.load(path, mmap_mode="r")
        return a.ndim == 3 and a.shape[1] == N_KEYPOINTS and a.shape[2] == 3
    except Exception:
        return False


def eligible_rows() -> list[dict]:
    """Rows with snippet video + SuperAnimal pose on disk (39 keypoints — not ViTPose 17)."""
    out: list[dict] = []
    for r in rows:
        sid = r.get("snippet_id")
        if not isinstance(sid, str):
            continue
        vp = r.get("video_path")
        if not vp or not (REPO_ROOT / vp).is_file():
            continue

        guess = POSE_OUT / f"{sid}_pose.npy"
        pose_abs: Path | None = None
        if guess.is_file() and _is_superanimal_pose_file(guess):
            pose_abs = guess
        else:
            pp = r.get("pose_path")
            if pp and (REPO_ROOT / pp).is_file() and _is_superanimal_pose_file(REPO_ROOT / pp):
                pose_abs = REPO_ROOT / pp

        if pose_abs is None:
            continue

        row = dict(r)
        row["pose_path"] = str(pose_abs.relative_to(REPO_ROOT)).replace("\\", "/")

        mask_abs = POSE_OUT / f"{sid}_pose_mask.npy"
        mp = r.get("pose_mask_path")
        if mask_abs.is_file():
            row["pose_mask_path"] = str(mask_abs.relative_to(REPO_ROOT)).replace("\\", "/")
        elif mp and (REPO_ROOT / mp).is_file():
            row["pose_mask_path"] = str((REPO_ROOT / mp).relative_to(REPO_ROOT)).replace("\\", "/")
        else:
            row["pose_mask_path"] = None

        out.append(row)
    return out


ELIGIBLE = eligible_rows()
print(f"Rows with video + SuperAnimal pose on disk: {len(ELIGIBLE)}")
if not ELIGIBLE:
    print(
        "No matches: run ``python dataset_construction/06_pose_extraction_superanimal.py`` "
        f"then check ``{POSE_OUT.relative_to(REPO_ROOT)}/{{snippet_id}}_pose.npy``"
    )

Rows with video + SuperAnimal pose on disk: 3


In [3]:
VIEW_MIN_JOINT_CONF = 0.25
# DeepLabCut ``.h5`` / our ``.npy`` use **x, y in pixel** space of the analyzed snippet (not 0..1).
SUPERANIMAL_XY_IS_PIXEL = True

JOINT_COLORS = [
    (0, 255, 255), (0, 200, 255), (0, 150, 255), (0, 100, 255), (0, 50, 255),
    (0, 255, 128), (0, 255, 0), (50, 255, 0), (100, 255, 0), (150, 255, 0),
    (255, 255, 0), (255, 200, 0), (255, 150, 0), (255, 100, 0), (255, 50, 0),
    (255, 0, 0), (255, 0, 50), (255, 0, 100), (255, 0, 150), (255, 0, 200),
    (255, 0, 255), (200, 0, 255), (150, 0, 255), (100, 0, 255), (50, 0, 255),
    (180, 255, 180), (180, 180, 255), (255, 180, 180), (255, 255, 180), (180, 255, 255),
    (255, 180, 255), (128, 128, 128), (200, 200, 200), (255, 255, 255), (100, 100, 100),
    (50, 50, 50), (10, 200, 100), (100, 10, 200), (200, 100, 10),
]

# From ``superquadruped_dataset.json`` (AdaptiveMotorControlLab/modelzoo-figures) — **not** APT36K.
SUPERANIMAL_QUADRUPED_EDGES: list[tuple[int, int]] = [
    (10, 5),
    (10, 11),
    (11, 12),
    (5, 6),
    (6, 7),
    (10, 0),
    (2, 4),
    (2, 3),
    (2, 1),
    (5, 0),
    (19, 21),
    (21, 20),
    (20, 22),
    (21, 38),
    (21, 37),
    (37, 36),
    (38, 36),
    (22, 23),
    (18, 24),
    (24, 25),
    (25, 26),
    (18, 27),
    (27, 28),
    (28, 29),
    (22, 31),
    (31, 33),
    (33, 30),
    (22, 32),
    (32, 34),
    (34, 35),
    (13, 14),
    (8, 9),
    (15, 16),
    (16, 19),
    (19, 27),
    (19, 24),
    (17, 18),
    (36, 38),
    (36, 37),
    (11, 6),
]
SKELETON_LINE_BGR = (220, 235, 255)


def dot_radius_for_canvas(width: int, height: int) -> int:
    return int(np.clip(min(width, height) // 48, 12, 28))


def xy_image_from_pose_row(row: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """DLC outputs (x, y, likelihood); no axis swapping."""
    xc = np.asarray(row[:, 0], dtype=np.float64)
    yr = np.asarray(row[:, 1], dtype=np.float64)
    return xc, yr


def coords_are_normalized(pose: np.ndarray, conf_thr: float = 0.08) -> bool:
    """Heuristic: unit-ish x,y vs pixel coordinates."""
    if SUPERANIMAL_XY_IS_PIXEL:
        return False
    conf = pose[:, :, 2]
    sel = np.isfinite(conf) & (conf >= conf_thr)
    xc = pose[:, :, 0]
    yr = pose[:, :, 1]
    if sel.any():
        vx, vy = xc[sel], yr[sel]
    else:
        m = np.isfinite(xc) & np.isfinite(yr)
        if not m.any():
            return True
        vx, vy = xc[m], yr[m]
    vmax = float(max(np.nanmax(np.abs(vx)), np.nanmax(np.abs(vy))))
    # DLC / SuperAnimal outputs are usually **pixel** x,y — any value >> 1 is not unit-normalized.
    if vmax > 2.0:
        return False
    return vmax <= 1.5


def joint_screen_positions(pose_row, width, height, normalized, conf_thr):
    if SUPERANIMAL_XY_IS_PIXEL:
        normalized = False
    n = min(N_KEYPOINTS, pose_row.shape[0])
    xc_all, yr_all = xy_image_from_pose_row(pose_row[:n])
    conf_all = pose_row[:n, 2]
    out = [None] * N_KEYPOINTS
    for k in range(n):
        xc, yr, c = float(xc_all[k]), float(yr_all[k]), float(conf_all[k])
        if not np.isfinite(xc) or not np.isfinite(yr) or c < conf_thr:
            continue
        if normalized:
            xi = int(np.clip(xc * width, 0, width - 1))
            yi = int(np.clip(yr * height, 0, height - 1))
        else:
            xi = int(np.clip(xc, 0, width - 1))
            yi = int(np.clip(yr, 0, height - 1))
        out[k] = (xi, yi)
    return out


def draw_pose_frame(pose_row, mask_ok, width, height, normalized, conf_thr=VIEW_MIN_JOINT_CONF, radius=None):
    img = np.zeros((height, width, 3), dtype=np.uint8)
    if not mask_ok:
        return img
    r = dot_radius_for_canvas(width, height) if radius is None else int(radius)
    pts = joint_screen_positions(pose_row, width, height, normalized, conf_thr)

    for a, b in SUPERANIMAL_QUADRUPED_EDGES:
        pa, pb = pts[a], pts[b]
        if pa is None or pb is None:
            continue
        cv2.line(img, pa, pb, SKELETON_LINE_BGR, max(3, r // 3), lineType=cv2.LINE_AA)

    for k in range(min(N_KEYPOINTS, pose_row.shape[0])):
        if pts[k] is None:
            continue
        color = JOINT_COLORS[k % len(JOINT_COLORS)]
        cv2.circle(img, pts[k], r + max(2, r // 5), (255, 255, 255), max(2, r // 5), lineType=cv2.LINE_AA)
        cv2.circle(img, pts[k], r, color, -1, lineType=cv2.LINE_AA)
    return img


def video_frame_count(path: Path) -> int:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return max(n, 0)


def fps_from_video(path: Path, fallback: float = 5.0) -> float:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return fallback
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 0.0
    cap.release()
    return fps if fps > 1e-3 else fallback


def video_size_from_first_frame(path: Path) -> tuple[int, int]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        raise RuntimeError(f"Cannot read first frame: {path}")
    hh, ww = frame.shape[:2]
    return max(ww, 1), max(hh, 1)


def map_pose_to_native_video_frames(
    pose: np.ndarray,
    mask: np.ndarray | None,
    n_vid: int,
    fps_video: float,
    pose_sample_hz: float,
) -> tuple[np.ndarray, np.ndarray]:
    t_pose, nk, nc = pose.shape
    if mask is None:
        mask = np.ones(t_pose, dtype=bool)
    else:
        mask = np.asarray(mask, dtype=bool)
    out = np.empty((n_vid, nk, nc), dtype=np.float32)
    valid = np.empty(n_vid, dtype=bool)
    if n_vid <= 0:
        return out.reshape(0, nk, nc), valid
    if fps_video <= 1e-9:
        fps_video = 25.0
    if pose_sample_hz <= 1e-9:
        pose_sample_hz = 5.0
    for fi in range(n_vid):
        t_sec = fi / fps_video
        j = int(np.clip(np.round(t_sec * pose_sample_hz), 0, t_pose - 1))
        out[fi] = pose[j]
        valid[fi] = bool(mask[j]) if j < len(mask) else False
    return out, valid


def render_pose_side_video(
    pose: np.ndarray,
    mask: np.ndarray | None,
    width: int,
    height: int,
    fps: float,
    out_path: Path,
    conf_thr: float = VIEW_MIN_JOINT_CONF,
) -> None:
    normalized = coords_are_normalized(pose)
    T = pose.shape[0]
    if mask is None:
        mask = np.ones(T, dtype=bool)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, float(max(fps, 1.0)), (width, height))
    if not writer.isOpened():
        raise RuntimeError("VideoWriter failed (codec); try another fourcc on your OS.")
    try:
        for t in range(T):
            ok = bool(mask[t]) if t < len(mask) else False
            frame = draw_pose_frame(pose[t], ok, width, height, normalized, conf_thr=conf_thr)
            writer.write(frame)
    finally:
        writer.release()

In [4]:
_LAST_TMP: Path | None = None


def random_snippet_and_pose(seed: int | None = None) -> tuple[str | None, str | None, str]:
    global _LAST_TMP
    if not ELIGIBLE:
        return (
            None,
            None,
            "No rows with snippet video + SuperAnimal pose .npy. "
            f"Expected under {POSE_OUT.relative_to(REPO_ROOT)}/",
        )
    if seed is not None:
        random.seed(seed)
    r = random.choice(ELIGIBLE)
    vid_p = REPO_ROOT / r["video_path"]
    pose_p = REPO_ROOT / r["pose_path"]
    mask_p = REPO_ROOT / r["pose_mask_path"] if r.get("pose_mask_path") else None

    pose = np.load(pose_p)
    mask = np.load(mask_p) if mask_p and mask_p.is_file() else None

    if pose.ndim != 3 or pose.shape[1] != N_KEYPOINTS or pose.shape[2] != 3:
        return None, None, f"Unexpected pose shape {pose.shape}; expected (T, {N_KEYPOINTS}, 3)"

    fps_v = fps_from_video(vid_p)
    n_vid = video_frame_count(vid_p)
    if n_vid <= 0:
        n_vid = int(max(round(pose.shape[0] * fps_v / POSE_TARGET_FPS), 1))

    w, h = video_size_from_first_frame(vid_p)
    pose_vis, mask_vis = map_pose_to_native_video_frames(
        pose,
        mask,
        n_vid=n_vid,
        fps_video=fps_v,
        pose_sample_hz=POSE_TARGET_FPS,
    )

    if _LAST_TMP is not None and _LAST_TMP.is_file():
        try:
            _LAST_TMP.unlink()
        except OSError:
            pass
    fd, tmp_path = tempfile.mkstemp(suffix="_superanimal_pose.mp4", prefix="gradio_sa_pose_")
    os.close(fd)
    out_tmp = Path(tmp_path)
    _LAST_TMP = out_tmp

    render_pose_side_video(pose_vis, mask_vis, w, h, fps_v, out_tmp)

    cap = (
        f"snippet_id={r.get('snippet_id')} | pose {pose.shape} → video {pose_vis.shape[0]} frames @ {fps_v:.2f} fps | "
        f"sampled @ {POSE_TARGET_FPS:g} Hz | {N_KEYPOINTS} kp (pixels) | "
        f"if skeleton looks wrong, re-extract with updated 06_pose_extraction_superanimal.py"
    )
    return str(vid_p.resolve()), str(out_tmp.resolve()), cap

In [ ]:
import gradio as gr

with gr.Blocks(title="Snippet vs SuperAnimal pose") as demo:
    gr.Markdown("### Original snippet (left) · SuperAnimal-Quadruped skeleton (right)")
    with gr.Row():
        v_left = gr.Video(label="Original snippet", interactive=False)
        v_right = gr.Video(label="Pose (39 keypoints)", interactive=False)
    cap = gr.Textbox(label="Info", interactive=False)
    seed_in = gr.Number(label="Random seed (optional, integer)", value=None, precision=0)
    btn = gr.Button("Load random snippet + pose", variant="primary")

    def _go(seed):
        try:
            s = int(seed) if seed is not None and str(seed).strip() != "" else None
            return random_snippet_and_pose(seed=s)
        except Exception as e:
            return None, None, f"Error: {e}"

    btn.click(_go, inputs=[seed_in], outputs=[v_left, v_right, cap])

demo.launch(share=False, inline=True, allowed_paths=[str(REPO_ROOT)])

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/gradio/components/video.py:398: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/Users/viktorzozula/Documents/Cats-in-Pain-Bachelors/.venv/lib/python3.13/site-packages/gradio/components/video.py:398: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
